<a href="https://colab.research.google.com/github/raulmabcn/DataAnalyst/blob/main/Foundations/Python/Exercices/Bloque2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###3️⃣ Pacientes – Análisis temporal

Calcula la estancia media (numero_dias_ingreso) por hospital, pero solo para pacientes menores de edad.

Encuentra los hospitales donde la estancia máxima supera en más de cinco veces la estancia media del mismo hospital.

Calcula el porcentaje de pacientes por hospital respecto al total nacional.

Detecta hospitales donde más del 10% de sus pacientes tengan menos de 15 años.

In [1]:
import pandas as pd

In [4]:
data_h = pd.read_csv('hospitals.csv', sep=';', encoding='latin-1')
data_h.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   hospital_id                 170 non-null    int64  
 1   nombre                      170 non-null    object 
 2   localidad                   170 non-null    object 
 3   provincia                   170 non-null    object 
 4   comunidad_autonoma          170 non-null    object 
 5   presupuesto_anual_millones  170 non-null    float64
 6   num_camas                   170 non-null    int64  
 7   num_medicos                 170 non-null    int64  
 8   indice_satisfaccion         170 non-null    float64
 9   porcentaje_ocupacion        170 non-null    float64
dtypes: float64(3), int64(3), object(4)
memory usage: 13.4+ KB


In [5]:
data_p = pd.read_csv('pacientes.csv', sep=';', encoding='latin-1')
data_p.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5060 entries, 0 to 5059
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   paciente_id          5060 non-null   int64 
 1   nombre               5060 non-null   object
 2   hospital_id          5060 non-null   int64 
 3   edad                 5060 non-null   int64 
 4   nacionalidad         5060 non-null   object
 5   numero_visitas       5060 non-null   int64 
 6   ingreso              5060 non-null   object
 7   numero_dias_ingreso  5060 non-null   int64 
dtypes: int64(5), object(3)
memory usage: 316.4+ KB


In [11]:
#mean_p_underage = data_p[(data_p['edad'] < 18)].groupby('hospital_id')['numero_dias_ingreso'].mean().round(2)
mean_p_underage =  data_p[(data_p['edad'] < 18)].groupby('hospital_id', as_index=False).agg(media_dias_pacientes =('numero_dias_ingreso', 'mean'))

data_h_p = data_h.merge( mean_p_underage, on='hospital_id', how='left')

#data_h_p[['nombre','numero_dias_ingreso']]
data_h_p[['nombre','media_dias_pacientes']]

,nombre,media_dias_pacientes
0,Hospital Universitario Torrecardenas,2.000000
1,Hospital de Poniente,8.000000
2,Hospital Universitario Puerta del Mar,4.333333
3,Hospital de Jerez,0.000000
4,Hospital Universitario Reina Sofia,10.000000
...,...,...
165,Hospital San Pedro,6.666667
166,Hospital Universitario de Ceuta,NaN
167,Hospital Comarcal de Melilla,NaN
168,Fundacion Hospital Calahorra,0.000000


In [27]:
hospital_n_dias_ingreso = data_p.groupby('hospital_id', as_index=False).agg(
    media_dias = ('numero_dias_ingreso', 'mean'),
    max_dias = ('numero_dias_ingreso', 'max')
)
data_h_p = hospital_n_dias_ingreso[ hospital_n_dias_ingreso['max_dias'] > (hospital_n_dias_ingreso['media_dias']*5) ].merge( data_h, on='hospital_id', how='left')
data_h_p[['nombre','max_dias','media_dias']]

,nombre,max_dias,media_dias
0,Hospital Universitario Torrecardenas,19,2.212121
1,Hospital Universitario Puerta del Mar,20,3.918919
2,Hospital de Jerez,17,3.333333
3,Hospital Universitario Reina Sofia,20,2.518519
4,Hospital Infanta Margarita,16,3.150000
...,...,...,...
81,Hospital Universitario Basurto,20,3.954545
82,Hospital Universitario de Cruces,19,2.029412
83,Hospital San Eloy,19,2.575758
84,Hospital Universitario de Ceuta,20,3.676471


In [29]:
total_patients = data_p['paciente_id'].count()
data_n_p = data_p.groupby('hospital_id').agg( numero_pacientes = ('paciente_id','count') )
data_n_p['prctj_pacientes'] = (data_n_p['numero_pacientes']/total_patients)*100
data_n_p

,numero_pacientes,prctj_pacientes
hospital_id,,
1,33,0.652174
2,33,0.652174
3,37,0.731225
4,33,0.652174
5,27,0.533597
...,...,...
166,36,0.711462
167,34,0.671937
168,47,0.928854


In [48]:
answer = pd.DataFrame({
    'total' : data_p.groupby('hospital_id')['paciente_id'].count(),
    'menores_15' : data_p[(data_p['edad'] < 15 )].groupby('hospital_id')['paciente_id'].count()
})

answer['ratio'] = answer['menores_15']/answer['total']
answer[answer['ratio'] > 0.10]

#total_patients = data_p.groupby('hospital_id').agg( total_patients = ('paciente_id','count'))
#patients_under_15 = data_p[(data_p['edad'] < 15 )].groupby('hospital_id').agg( patients_under_15 = ('paciente_id','count'))

#(( patients_under_15['patients_under_15']/total_patients['total_patients'] ) > 0.10).loc[lambda x:x]


,total,menores_15,ratio
hospital_id,,,
30,29,3.0,0.103448
32,27,3.0,0.111111
56,17,2.0,0.117647
60,23,3.0,0.130435
66,34,4.0,0.117647
72,33,5.0,0.151515
126,24,3.0,0.125000


###4️⃣ Especialidades – Cruces complejos

Muestra qué especialidad fija tiene mayor número de hospitales asociados.

Encuentra las especialidades donde todos los hospitales asociados tienen más de 60 camas.

In [49]:
data_e = pd.read_csv('especialidades.csv', sep=';', encoding='latin-1')
data_e.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3222 entries, 0 to 3221
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   hospital_id   3222 non-null   int64 
 1   especialidad  3222 non-null   object
 2   fija          3222 non-null   object
dtypes: int64(1), object(2)
memory usage: 75.6+ KB


In [60]:
#data_e[(data_e['fija'] == 'S')].groupby('especialidad')['hospital_id'].count().sort_values( ascending=False)
data_e[(data_e['fija'] == 'S')].groupby('especialidad')['hospital_id'].count().idxmax()

#data_e[data_e['fija'] == 'S']['especialidad'].value_counts().idxmax()


'Nefrologia'

In [73]:
def enough_beds( serie_num_beds):
  return (serie_num_beds > 60).all()

data_e_num_camas = data_e.merge( data_h[['hospital_id','num_camas']],on='hospital_id', how='left' )
answer = data_e_num_camas.groupby('especialidad').agg(
    #condition = ( 'num_camas', enough_beds )
    condition = ( 'num_camas', lambda x: (x > 60).all() )
)
answer[answer['condition']]

,condition
especialidad,
Dermatologia,True
Laboratorio Clinico,True
Medicina Nuclear,True
Medicina de Urgencias,True
Patologia,True
Psicologia Clinica,True
Unidad de Dolor,True
